## If working in Google Colab, run this first
Run the cell below at the start of every session. It downloads the course files to your Google Drive and keeps them up to date.
When you see "Setup complete", you are ready to run the rest of the notebook.

**Do not edit or save your work inside the `AIforMovementScience` folder.**
Use **File > Save a copy in Drive** and work on your own copy.

In [ ]:
# [course-setup] Run this cell first, every session (or use Runtime > Run all)
import os, sys
try:
    from google.colab import drive
    os.chdir('/content')                  # start from a valid directory
    drive.mount('/content/drive')
    REPO = '/content/drive/MyDrive/AIforMovementScience'
    if os.path.isdir(REPO + '/.git'):
        # already downloaded before: just update it
        !git -C "$REPO" fetch origin main && git -C "$REPO" reset --hard origin/main
    else:
        # first time, or a broken/partial copy: download a fresh copy
        !rm -rf "$REPO"
        !git clone https://github.com/dguari1/AIforMovementScience.git "$REPO"
except ImportError:
    # not in Colab: walk up from here until the repository root is found
    REPO = os.getcwd()
    while REPO != '/' and not os.path.isdir(os.path.join(REPO, 'lectures', 'data')):
        REPO = os.path.dirname(REPO)

WORKDIR = REPO + '/lectures'
os.chdir(WORKDIR)
sys.path.insert(0, WORKDIR)
print('Setup complete. You can now run the rest of the notebook.')

# Convolutional Neural Networks

In 05.02a, we applied different kernels to a signal to transform it in different ways.

A Convolutional Neural Network (CNN) applies many **kernels** to the signal, and makes many filtered versions of the signal (called **feature maps**). An *activation function* (a nonlinearity) then processes each feature map. A downsampling step (a stride larger than 1, or a **pooling** layer) makes the signals shorter. The network repeats this process in many layers.

<div style="text-align:center">
<img src="images/Schematic-of-1DCNN.jpg" alt="plot" width="400" />
</div>

At the end of the convolution layers, the network makes one long vector called the **embedding**. The embedding represents the input signal after the CNN transformed it. The embedding is the input of a *fully connected* (FC) layer, which is a regular ANN (as in 05.01). The output of the FC layer depends on the problem: a probability for each class in classification, or a number in regression.

The kernels are **not** chosen by us (as in 05.02a). The network learns them from the training data with gradient descent (or a modified version of it). We choose only how many kernels and how many layers to use.

When we create the CNN, the kernels get random values (*random initialization*). Training then changes these values.

In this notebook we build a small CNN with the **PyTorch** library (<https://pytorch.org>). You do not need to learn all of PyTorch. Focus on the main ideas.

## Learning goals
- understand the shape of the input of a 1D CNN: (samples, channels, length)
- see that a convolution layer contains learnable kernels
- follow a signal through the layers of a small CNN
- train a CNN with a training loop, and evaluate it on a test set
- look at the kernels after training

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn as nn

torch.manual_seed(0)   # same random kernels every time you run the notebook
np.random.seed(0)

Let us start with one layer. It receives one channel (one signal) and produces two signals: it applies two kernels to the input signal.

We use a kernel size of 3, as in 05.02a.

In [ ]:
conv =  nn.Conv1d(
    in_channels=1,
    out_channels=2,
    kernel_size=3,
    stride=1,
    padding=1,
    bias=True
)

print(conv)
print('Weight shape:', conv.weight.shape)
print('Bias shape:', conv.bias.shape)

We can plot the kernels (remember that these are random numbers).

In [ ]:
weights = conv.weight.detach().numpy()

plt.figure(figsize=(8, 3))
for i in range(weights.shape[0]):
    plt.plot(weights[i, 0, :], marker='o', label=f'Filter {i+1}')
plt.title('Randomly initialized convolution kernels')
plt.xlabel('Kernel index')
plt.ylabel('Weight value')
plt.grid(True)
plt.legend()
plt.show()

We can also plot the feature map made by each kernel.

In [ ]:
x = np.array([0, 0, 1, 3, 7, 8, 7, 3, 1, 0, 0], dtype=float)
x_torch = torch.tensor(x, dtype=torch.float32).view(1, 1, -1)  # (batch, channels, length)

with torch.no_grad():
    y_torch = conv(x_torch)


fig, ax = plt.subplots(3,1);
ax[0].plot(x,label='Input signal')
ax[0].legend()
ax[1].plot(y_torch.cpu().detach().numpy()[0][0], label='Output of first kernel')
ax[1].legend()
ax[2].plot(y_torch.cpu().detach().numpy()[0][1], label='Output of second kernel')
ax[2].legend()


Each feature map comes from a different kernel. Different kernels can find different patterns in the signal.

In a CNN, a nonlinear activation function (for example ReLU, tanh or leaky ReLU) processes each feature map. Then another convolution layer receives the feature maps and makes new feature maps.

The number of parameters grows fast. This first layer has 8 parameters (3 weights for each kernel, plus 1 bias for each kernel). A second layer with 2 kernels that each receive these 2 feature maps has 2 × (2 × 3) + 2 = 14 more parameters.

### A simple example

We create a small CNN to classify signals into two classes:
- layer 1: 4 kernels, so 4 feature maps
- layer 2: 8 kernels, each one combines the 4 feature maps, so 8 feature maps
- **global average pooling**: the mean of each feature map, so a vector of 8 numbers (the embedding)
- a fully connected layer that changes the 8 numbers into 2 class scores

Remember that the kernels start with random values. Training changes them.

In [ ]:
class TinyCNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv1d(in_channels=1, out_channels=4, kernel_size=3, padding=1),   # layer 1: 4 kernels
            nn.ReLU(),
            nn.Conv1d(in_channels=4, out_channels=8, kernel_size=3, padding=1),   # layer 2: 8 kernels
            nn.ReLU(),
            nn.AdaptiveAvgPool1d(1)                                               # mean of each feature map
        )
        self.classifier = nn.Linear(8, 2)    # 8 numbers -> 2 class scores

    def forward(self, x):
        x = self.features(x)      # shape (batch, 8, 1)
        x = x.squeeze(-1)         # shape (batch, 8): the embedding
        x = self.classifier(x)    # shape (batch, 2)
        return x

model = TinyCNN()
print(model)
print('Number of parameters:', sum(p.numel() for p in model.parameters()))

In [ ]:
model.eval()   # evaluation mode: we only use the model, we do not train it

with torch.no_grad():
    stages = [x_torch.squeeze(0)]
    stage_names = ["Input"]
    z_vis = x_torch.clone()
    for li, lyr in enumerate(model.features):
        z_vis = lyr(z_vis)
        stages.append(z_vis.squeeze(0))
        stage_names.append(f"Layer {li}: {lyr.__class__.__name__}")

for name, act in zip(stage_names, stages):
    if act.shape[-1] == 1:    # after pooling, each channel is one number
        plt.figure(figsize=(6, 2.5))
        plt.bar(range(act.shape[0]), act[:, 0].numpy())
        plt.xlabel('Channel')
        plt.title(name + ' (the embedding)')
        plt.show()
        continue
    n_channels = act.shape[0]
    fig, axes = plt.subplots(n_channels, 1, figsize=(9, 1.5 * n_channels), sharex=True, squeeze=False)
    for ch in range(n_channels):
        axes[ch, 0].plot(act[ch].numpy(), linewidth=2)
        axes[ch, 0].set_ylabel(f"ch {ch}")
        axes[ch, 0].grid(True, alpha=0.3)
    axes[0, 0].set_title(name)
    axes[-1, 0].set_xlabel("Time index")
    plt.tight_layout()
    plt.show()

with torch.no_grad():
    logits = model(x_torch)                    # class scores
    probs = torch.softmax(logits, dim=1)       # class probabilities (softmax, as in lecture 04.01)
print('Class scores (logits):', logits.numpy().round(3))
print('Class probabilities:  ', probs.numpy().round(3))

plt.figure(figsize=(5, 3))
plt.bar(["Class 0", "Class 1"], probs.squeeze(0).numpy())
plt.ylim(0, 1)
plt.ylabel("Probability")
plt.title("Predicted class probabilities (random kernels)")
plt.show()

With random kernels, the probabilities are close to 0.5: the network does not know anything yet.

Now we train the network. Class 1 has signals with a local peak. Class 0 has signals with noise only. **The peak is at a random position** in each signal. This is a simple problem, but a good example: a kernel can find a pattern at any position in the signal.

We make 200 signals. As in module 04, we keep part of the data for the test.

In [ ]:
n_samples = 200
length = 40
peak = np.array([0.5, 1.2, 1.2, 0.5])
X, y = [], []

for i in range(n_samples):
    signal = np.random.normal(0, 0.4, length)
    if i % 2 == 0:                               # class 1: add a peak at a random position
        start = np.random.randint(0, length - len(peak))
        signal[start:start + len(peak)] += peak
        y.append(1)
    else:                                        # class 0: noise only
        y.append(0)
    X.append(signal)

X = np.array(X, dtype=np.float32)
y = np.array(y, dtype=np.int64)

from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=0, stratify=y)

# shape for Conv1d: (samples, channels, length)
X_train_t = torch.tensor(X_train).unsqueeze(1)
X_test_t = torch.tensor(X_test).unsqueeze(1)
y_train_t = torch.tensor(y_train)
y_test_t = torch.tensor(y_test)
print('Train:', X_train_t.shape, ' Test:', X_test_t.shape)

In [ ]:
idx_peak = np.where(y_train == 1)[0]
idx_noise = np.where(y_train == 0)[0]

fig, axes = plt.subplots(1, 2, figsize=(12, 4), sharey=True)
for k in idx_peak[:5]:
    axes[0].plot(X_train[k], alpha=0.7)
axes[0].set_title('Class 1: a peak at a random position (5 examples)')
for k in idx_noise[:5]:
    axes[1].plot(X_train[k], alpha=0.7)
axes[1].set_title('Class 0: noise only (5 examples)')
for a in axes:
    a.set_xlabel('Time index')
    a.grid(True, alpha=0.3)
axes[0].set_ylabel('Amplitude')
plt.tight_layout()
plt.show()

We train the CNN with the **cross-entropy loss**. This is the same loss as in logistic regression (lecture 04.01), extended to many classes.

The training loop repeats these steps. One repetition is one **epoch** (one pass through all the training data, as in 05.01):
1. `model(X)`: compute the class scores
2. `criterion(...)`: compute the loss
3. `loss.backward()`: compute the gradient of the loss for each parameter
4. `optimizer.step()`: change the parameters a small step against the gradient (gradient descent)

We use the test set **only to show** how the test accuracy changes. To **choose** the number of epochs, we must use a validation set (we do this in 05.02c).

In [ ]:
torch.manual_seed(0)
model = TinyCNN()
criterion = nn.CrossEntropyLoss()
# Adam: a modified version of gradient descent that adapts the step size of each parameter
optimizer = torch.optim.Adam(model.parameters(), lr=0.01)

history = {'train_loss': [], 'train_acc': [], 'test_acc': []}
for epoch in range(150):
    model.train()                          # training mode
    optimizer.zero_grad()
    logits = model(X_train_t)
    loss = criterion(logits, y_train_t)
    loss.backward()
    optimizer.step()

    model.eval()                           # evaluation mode
    with torch.no_grad():
        train_acc = (model(X_train_t).argmax(dim=1) == y_train_t).float().mean().item()
        test_acc = (model(X_test_t).argmax(dim=1) == y_test_t).float().mean().item()
    history['train_loss'].append(loss.item())
    history['train_acc'].append(train_acc)
    history['test_acc'].append(test_acc)
    if (epoch + 1) % 25 == 0:
        print(f'Epoch {epoch+1:3d} | Loss = {loss.item():.4f} | Train accuracy = {train_acc:.3f} | Test accuracy = {test_acc:.3f}')

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(12, 4))
ax[0].plot(history['train_loss'])
ax[0].set_title('Training loss')
ax[0].set_xlabel('Epoch')
ax[0].set_ylabel('Cross-entropy loss')
ax[1].plot(history['train_acc'], label='Train')
ax[1].plot(history['test_acc'], label='Test')
ax[1].set_title('Accuracy')
ax[1].set_xlabel('Epoch')
ax[1].set_ylim(0.4, 1.02)
ax[1].legend()
for a in ax:
    a.grid(True, alpha=0.3)
plt.show()

## What did the network learn?

The first layer has 4 kernels of 3 points. Let us plot them after training, and apply them to one signal with a peak. Compare them with the kernels of 05.02a (smoothing, edge detector).

In [ ]:
w1 = model.features[0].weight.detach().numpy()     # shape (4, 1, 3)

fig, ax = plt.subplots(1, 2, figsize=(13, 4))
for k in range(w1.shape[0]):
    ax[0].plot(w1[k, 0], marker='o', label=f'Kernel {k}: {np.round(w1[k, 0], 2)}')
ax[0].set_title('Layer 1 kernels after training')
ax[0].set_xlabel('Kernel index')
ax[0].legend(fontsize=8)
ax[0].grid(True, alpha=0.3)

example = X_test_t[y_test_t == 1][:1]               # one test signal with a peak
with torch.no_grad():
    maps = torch.relu(model.features[0](example))[0].numpy()
ax[1].plot(example[0, 0].numpy(), 'k', linewidth=2, label='Input')
for k in range(maps.shape[0]):
    ax[1].plot(maps[k], label=f'Feature map {k} (after ReLU)')
ax[1].set_title('Layer 1 feature maps for one test signal with a peak')
ax[1].set_xlabel('Time index')
ax[1].legend(fontsize=8)
ax[1].grid(True, alpha=0.3)
plt.show()

Some kernels have all positive weights, similar to a smoothing kernel: their feature map is large where the peak is, at any position. A kernel with all negative weights does the opposite: its feature map is small at the peak. The network found useful kernels from the data, without our help. (Your kernels can be different if you use a different computer or a GPU.)

Key points:
- A CNN learns its kernels from the data.
- The same kernel is applied at all positions, so the CNN can find a pattern at any position in the signal.
- As for any model, we must evaluate the CNN on data that it did not see during training.

In 05.02c, we use a CNN on real sensor data.